[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/bike-availability-data-science-full/blob/main/notebooks/Module_02_Data_Acquisition/M2_04_merge_datasets.ipynb)

# 🔗 Module 02: Merge Bike and Weather Datasets

**Purpose**: Learn to merge datasets with different time granularities and handle time alignment  
**Module**: Module 02 - Data Acquisition  
**Author**: [Your Name]  
**Date**: 2026-01-15

---

## 📋 Overview

In this notebook, you will:
- [ ] Load bike and weather datasets
- [ ] Understand time alignment challenges
- [ ] Implement time-based merging strategies
- [ ] Handle missing data and gaps
- [ ] Create a unified analysis-ready dataset
- [ ] Validate the merged data
- [ ] Save the final integrated dataset

**Goal**: Master time series data integration!

**Estimated Time**: 120-150 minutes

---

## 🎓 Learning Path Note

**This Module**: Learn data merging strategies and time alignment through hands-on experimentation in notebooks.

**Module 8 - Automation**: You'll convert these merge strategies into automated pipelines. The `src/data_acquisition.py` module contains a `merge_bike_weather()` function as a reference implementation.

**Learning Strategy**: 
- ✅ DO: Explore different merge strategies and understand their trade-offs
- ⏸️ LATER: Build automated ETL pipelines using these patterns (Module 8)

---

---

## 📖 Part 1: The Time Alignment Challenge

### Why Merging is Difficult

When combining datasets, you often face:

**1. Different Time Granularities**
- Bike data: Snapshots every 5-10 minutes
- Weather data: Hourly measurements
- **Solution**: Aggregate or interpolate

**2. Missing Timestamps**
- API downtime
- Data collection gaps
- **Solution**: Forward fill, interpolation, or drop

**3. Timezone Issues**
- Data from different sources may use different timezones
- **Solution**: Standardize to UTC or local time

**4. Mismatched Keys**
- One dataset has location IDs, another has coordinates
- **Solution**: Join on multiple criteria or nearest match

### Merge Strategies

| Strategy | When to Use | Pros | Cons |
|----------|-------------|------|------|
| **Exact Match** | Same time granularity | Perfect alignment | Strict, loses data |
| **Nearest Time** | Different granularities | Flexible | Approximate |
| **Aggregate** | Higher → Lower granularity | Accurate summary | Loses detail |
| **Interpolate** | Lower → Higher granularity | Smooth | Imputes data |


---

## 🔧 Part 2: Setup

Run this cell first to set up the environment.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. IMPORTS
# ═══════════════════════════════════════════════════════════

# Standard library
import os
import sys
import json
from datetime import datetime, timedelta
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import DateFormatter

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    print("📍 Running in Google Colab")
else:
    print("📍 Running locally")

# Add project root to path
project_root = os.path.abspath('../..' if 'notebooks' in os.getcwd() else '.')
if project_root not in sys.path:
    sys.path.append(project_root)

print("✅ Setup complete!")
print(f"📁 Working directory: {os.getcwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## ⚙️ Part 3: Create Sample Datasets

For demonstration, let's create sample bike and weather datasets with realistic patterns.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. CREATE SAMPLE DATA
# ═══════════════════════════════════════════════════════════

np.random.seed(42)

# Create bike data (every 10 minutes for 7 days)
print("🚴 Creating bike availability data...")
bike_timestamps = pd.date_range('2026-01-01', periods=7*24*6, freq='10min')

bike_data_list = []
for ts in bike_timestamps:
    # Simulate 3 stations
    for station_id in ['STN_001', 'STN_002', 'STN_003']:
        bikes = max(0, int(10 + 5 * np.sin(ts.hour / 24 * 2 * np.pi) + np.random.normal(0, 2)))
        bike_data_list.append({
            'timestamp': ts,
            'station_id': station_id,
            'bikes_available': bikes,
            'docks_available': 20 - bikes
        })

df_bikes = pd.DataFrame(bike_data_list)
print(f"✅ Created bike data: {len(df_bikes):,} rows")
print(f"   Time range: {df_bikes['timestamp'].min()} to {df_bikes['timestamp'].max()}")
print(f"   Frequency: Every 10 minutes")

# Create weather data (hourly for 7 days)
print("\n🌦️ Creating weather data...")
weather_timestamps = pd.date_range('2026-01-01', periods=7*24, freq='H')

weather_data_list = []
for ts in weather_timestamps:
    weather_data_list.append({
        'timestamp': ts,
        'temperature_c': 10 + 5 * np.sin(ts.hour / 24 * 2 * np.pi) + np.random.normal(0, 2),
        'precipitation_mm': max(0, np.random.exponential(0.3)),
        'wind_speed_kmh': max(0, 15 + np.random.normal(0, 5)),
        'humidity_pct': min(100, max(0, 70 + np.random.normal(0, 10)))
    })

df_weather = pd.DataFrame(weather_data_list)
print(f"✅ Created weather data: {len(df_weather):,} rows")
print(f"   Time range: {df_weather['timestamp'].min()} to {df_weather['timestamp'].max()}")
print(f"   Frequency: Hourly")

# Introduce some realistic issues
# 1. Add some missing timestamps in bike data
missing_indices = np.random.choice(df_bikes.index, size=50, replace=False)
df_bikes = df_bikes.drop(missing_indices).reset_index(drop=True)
print(f"\n⚠️ Simulated issues:")
print(f"   Removed 50 bike data records (simulating API downtime)")

# 2. Add some missing values in weather data
weather_missing_indices = np.random.choice(df_weather.index, size=5, replace=False)
df_weather.loc[weather_missing_indices, 'temperature_c'] = np.nan
print(f"   Added 5 missing temperature values")

print("\n" + "=" * 60)
print("📊 Data Summary:")
print("=" * 60)
print(f"Bike data: {len(df_bikes):,} rows, {len(df_bikes['station_id'].unique())} stations")
print(f"Weather data: {len(df_weather):,} rows")
print(f"Time overlap: Both cover 7 days (2026-01-01 to 2026-01-07)")

### Inspect the Data

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. INSPECT DATA
# ═══════════════════════════════════════════════════════════

print("🚴 Bike Data Sample:")
print("=" * 60)
display(df_bikes.head(10))

print("\n🌦️ Weather Data Sample:")
print("=" * 60)
display(df_weather.head(10))

# Check time distributions
print("\n" + "=" * 60)
print("⏰ Time Granularity Analysis:")
print("=" * 60)

bike_time_diffs = df_bikes.groupby('station_id')['timestamp'].diff().dropna()
print(f"Bike data time differences (per station):")
print(f"  Most common: {bike_time_diffs.mode().iloc[0]}")
print(f"  Range: {bike_time_diffs.min()} to {bike_time_diffs.max()}")

weather_time_diffs = df_weather['timestamp'].diff().dropna()
print(f"\nWeather data time differences:")
print(f"  Most common: {weather_time_diffs.mode().iloc[0]}")
print(f"  All hourly: {(weather_time_diffs == pd.Timedelta(hours=1)).all()}")

---

## 🔗 Part 4: Merge Strategy 1 - Nearest Time Match

Match each bike record to the nearest weather timestamp.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. MERGE STRATEGY 1: NEAREST TIME
# ═══════════════════════════════════════════════════════════

def merge_nearest_time(df_main, df_lookup, time_col='timestamp', 
                       tolerance='30min', direction='nearest'):
    """
    Merge datasets using nearest time match.
    
    Parameters:
    -----------
    df_main : pandas.DataFrame
        Main dataset with timestamps
    df_lookup : pandas.DataFrame
        Lookup dataset with timestamps
    time_col : str
        Name of timestamp column
    tolerance : str
        Maximum time difference to allow
    direction : str
        'nearest', 'forward', or 'backward'
    
    Returns:
    --------
    pandas.DataFrame : Merged dataset
    """
    # Ensure timestamp columns are datetime
    df_main = df_main.copy()
    df_lookup = df_lookup.copy()
    
    df_main[time_col] = pd.to_datetime(df_main[time_col])
    df_lookup[time_col] = pd.to_datetime(df_lookup[time_col])
    
    # Sort both datasets by time
    df_main = df_main.sort_values(time_col)
    df_lookup = df_lookup.sort_values(time_col)
    
    # Use pandas merge_asof for nearest time matching
    merged = pd.merge_asof(
        df_main,
        df_lookup,
        on=time_col,
        direction=direction,
        tolerance=pd.Timedelta(tolerance)
    )
    
    return merged


# Test nearest time merge
print("🔗 Merging with nearest time strategy...")
print(f"   Bike data: {len(df_bikes):,} rows")
print(f"   Weather data: {len(df_weather):,} rows")

df_merged_nearest = merge_nearest_time(
    df_bikes, 
    df_weather,
    tolerance='30min',
    direction='nearest'
)

print(f"✅ Merged dataset: {len(df_merged_nearest):,} rows")
print(f"\n📊 Sample of merged data:")
display(df_merged_nearest.head(10))

# Check for missing weather data
weather_cols = ['temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'humidity_pct']
missing_weather = df_merged_nearest[weather_cols].isna().any(axis=1).sum()
print(f"\n⚠️ Records with missing weather data: {missing_weather}")

### Visualize the Merge

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. VISUALIZE MERGE QUALITY
# ═══════════════════════════════════════════════════════════

# Select one station for visualization
station = 'STN_001'
df_station = df_merged_nearest[df_merged_nearest['station_id'] == station].copy()

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
fig.suptitle(f'🔗 Merged Data Visualization - {station}', fontsize=16, fontweight='bold')

# Plot 1: Bikes available
axes[0].plot(df_station['timestamp'], df_station['bikes_available'], 
             color='steelblue', linewidth=1.5, marker='o', markersize=2)
axes[0].set_ylabel('Bikes Available')
axes[0].set_title('Bike Availability (10-min intervals)')
axes[0].grid(True, alpha=0.3)

# Plot 2: Temperature
axes[1].plot(df_station['timestamp'], df_station['temperature_c'], 
             color='coral', linewidth=2, marker='s', markersize=3)
axes[1].set_ylabel('Temperature (°C)')
axes[1].set_title('Temperature (Hourly, matched to bike data)')
axes[1].grid(True, alpha=0.3)

# Plot 3: Precipitation
axes[2].bar(df_station['timestamp'], df_station['precipitation_mm'], 
            width=0.01, color='steelblue', alpha=0.7)
axes[2].set_ylabel('Precipitation (mm)')
axes[2].set_xlabel('Time')
axes[2].set_title('Precipitation (Hourly)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Visualization shows how hourly weather data is matched to 10-min bike data")

---

## 🧮 Part 5: Merge Strategy 2 - Aggregate Bike Data

Aggregate bike data to hourly to match weather frequency.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. MERGE STRATEGY 2: AGGREGATE
# ═══════════════════════════════════════════════════════════

def aggregate_to_hourly(df, time_col='timestamp', agg_funcs=None):
    """
    Aggregate data to hourly frequency.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Dataset to aggregate
    time_col : str
        Name of timestamp column
    agg_funcs : dict
        Aggregation functions for each column
    
    Returns:
    --------
    pandas.DataFrame : Aggregated dataset
    """
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    
    # Set default aggregation functions if not provided
    if agg_funcs is None:
        agg_funcs = {
            'bikes_available': 'mean',
            'docks_available': 'mean'
        }
    
    # Group by station and hour, then aggregate
    df['hour'] = df[time_col].dt.floor('H')
    
    grouped_cols = ['hour', 'station_id'] if 'station_id' in df.columns else ['hour']
    
    df_agg = df.groupby(grouped_cols).agg(agg_funcs).reset_index()
    df_agg = df_agg.rename(columns={'hour': time_col})
    
    # Round values
    for col in df_agg.columns:
        if df_agg[col].dtype in ['float64', 'float32']:
            df_agg[col] = df_agg[col].round(2)
    
    return df_agg


# Aggregate bike data to hourly
print("🧮 Aggregating bike data to hourly frequency...")
df_bikes_hourly = aggregate_to_hourly(
    df_bikes,
    agg_funcs={
        'bikes_available': 'mean',  # Average bikes per hour
        'docks_available': 'mean'   # Average docks per hour
    }
)

print(f"✅ Original bike data: {len(df_bikes):,} rows (10-min intervals)")
print(f"✅ Aggregated bike data: {len(df_bikes_hourly):,} rows (hourly)")
print(f"\n📊 Sample of hourly bike data:")
display(df_bikes_hourly.head(10))

# Now merge on exact timestamps
print("\n🔗 Merging hourly datasets with exact timestamp match...")
df_merged_hourly = pd.merge(
    df_bikes_hourly,
    df_weather,
    on='timestamp',
    how='inner'  # Only keep records with both bike and weather data
)

print(f"✅ Merged dataset: {len(df_merged_hourly):,} rows")
print(f"\n📊 Sample of merged hourly data:")
display(df_merged_hourly.head(10))

# Check for missing data
print(f"\n🔍 Data Quality Check:")
print(f"   Missing values per column:")
print(df_merged_hourly.isna().sum())

### 🧠 Learner Task 1: Compare Merge Strategies

**Your Task**: Compare the two merge strategies and analyze the trade-offs.

Complete the analysis below:

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. LEARNER TASK 1: COMPARE STRATEGIES
# ═══════════════════════════════════════════════════════════

print("📊 Merge Strategy Comparison:")
print("=" * 60)

# TODO: Compare row counts
print(f"Strategy 1 (Nearest Time): {len(df_merged_nearest):,} rows")
print(f"Strategy 2 (Aggregate):    {len(df_merged_hourly):,} rows")

# TODO: Compare data granularity
print(f"\nData Granularity:")
print(f"Strategy 1: 10-minute bike data, hourly weather")
print(f"Strategy 2: Hourly bike data, hourly weather")

# TODO: Calculate average bikes for one station using both strategies
station = 'STN_001'
# Hint: Use df_merged_nearest and df_merged_hourly

avg_nearest = df_merged_nearest[df_merged_nearest['station_id']==station]['bikes_available'].mean()
avg_hourly = df_merged_hourly[df_merged_hourly['station_id']==station]['bikes_available'].mean()

print(f"\nAverage bikes available ({station}):")
print(f"Strategy 1 (Nearest): {avg_nearest:.2f}")
print(f"Strategy 2 (Aggregate): {avg_hourly:.2f}")

# TODO: Identify trade-offs
print(f"\n💡 Trade-offs:")
print(f"Strategy 1 Pros:")
print(f"  ✅ Keeps original time resolution")
print(f"  ✅ More data points for analysis")
print(f"Strategy 1 Cons:")
print(f"  ❌ Weather data repeated for multiple bike records")
print(f"  ❌ Larger dataset")

print(f"\nStrategy 2 Pros:")
print(f"  ✅ Clean 1:1 time alignment")
print(f"  ✅ Smaller, more manageable dataset")
print(f"  ✅ Averages smooth out noise")
print(f"Strategy 2 Cons:")
print(f"  ❌ Loses detailed bike availability patterns")
print(f"  ❌ May miss short-term fluctuations")

---

## 🧹 Part 6: Handle Missing Data

Deal with missing values in the merged dataset.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 8. HANDLE MISSING DATA
# ═══════════════════════════════════════════════════════════

# Let's use the hourly merged data going forward
df_final = df_merged_hourly.copy()

print("🧹 Handling missing data...")
print("=" * 60)

# Check missing values
print("Missing values before cleaning:")
missing_before = df_final.isna().sum()
print(missing_before[missing_before > 0])

# Strategy 1: Forward fill for temperature (reasonable for short gaps)
df_final['temperature_c'] = df_final.groupby('station_id')['temperature_c'].fillna(method='ffill')

# Strategy 2: Fill remaining with interpolation
df_final['temperature_c'] = df_final.groupby('station_id')['temperature_c'].interpolate(method='linear')

# Strategy 3: Fill any remaining with median
for col in ['precipitation_mm', 'wind_speed_kmh', 'humidity_pct']:
    if df_final[col].isna().any():
        median_val = df_final[col].median()
        df_final[col] = df_final[col].fillna(median_val)
        print(f"Filled {col} with median: {median_val:.2f}")

print("\n✅ Missing values after cleaning:")
missing_after = df_final.isna().sum()
if missing_after.sum() == 0:
    print("No missing values!")
else:
    print(missing_after[missing_after > 0])

print(f"\n📊 Final dataset: {len(df_final):,} rows, {len(df_final.columns)} columns")

---

## ✅ Part 7: Data Validation

Validate the merged dataset before saving.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 9. VALIDATE MERGED DATA
# ═══════════════════════════════════════════════════════════

def validate_merged_dataset(df):
    """
    Comprehensive validation of merged bike-weather dataset.
    
    Returns:
    --------
    bool : True if all validations pass
    """
    validations = []
    
    print("🔍 Validating merged dataset...")
    print("=" * 60)
    
    # Check 1: Required columns
    required_cols = ['timestamp', 'station_id', 'bikes_available', 
                    'temperature_c', 'precipitation_mm']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"❌ Missing required columns: {missing_cols}")
        validations.append(False)
    else:
        print(f"✅ All required columns present")
        validations.append(True)
    
    # Check 2: No missing values
    if df.isna().any().any():
        missing_count = df.isna().sum().sum()
        print(f"❌ Found {missing_count} missing values")
        validations.append(False)
    else:
        print(f"✅ No missing values")
        validations.append(True)
    
    # Check 3: Timestamps are continuous (hourly)
    for station in df['station_id'].unique():
        station_data = df[df['station_id'] == station].sort_values('timestamp')
        time_diffs = station_data['timestamp'].diff().dropna()
        expected_diff = pd.Timedelta(hours=1)
        
        if not (time_diffs == expected_diff).all():
            gaps = (time_diffs != expected_diff).sum()
            print(f"⚠️ Station {station}: Found {gaps} time gaps")
            validations.append(True)  # Warning, not error
        else:
            print(f"✅ Station {station}: Continuous hourly data")
            validations.append(True)
    
    # Check 4: Values in reasonable ranges
    checks = [
        ('bikes_available', 0, 100, "Bikes available"),
        ('temperature_c', -50, 50, "Temperature"),
        ('precipitation_mm', 0, 100, "Precipitation"),
        ('wind_speed_kmh', 0, 200, "Wind speed"),
        ('humidity_pct', 0, 100, "Humidity")
    ]
    
    for col, min_val, max_val, name in checks:
        if col in df.columns:
            out_of_range = ((df[col] < min_val) | (df[col] > max_val)).sum()
            if out_of_range > 0:
                print(f"⚠️ {name}: {out_of_range} values out of range [{min_val}, {max_val}]")
            else:
                print(f"✅ {name}: All values in valid range")
            validations.append(True)
    
    # Check 5: Sufficient data volume
    min_days = 1
    actual_days = (df['timestamp'].max() - df['timestamp'].min()).days
    if actual_days < min_days:
        print(f"⚠️ Warning: Only {actual_days} days of data (minimum {min_days} recommended)")
    else:
        print(f"✅ Sufficient data: {actual_days} days")
        validations.append(True)
    
    # Check 6: Multiple stations
    num_stations = df['station_id'].nunique()
    if num_stations < 2:
        print(f"⚠️ Warning: Only {num_stations} station(s) in dataset")
    else:
        print(f"✅ Multiple stations: {num_stations} stations")
        validations.append(True)
    
    print("=" * 60)
    
    return all(validations)


# Run validation
is_valid = validate_merged_dataset(df_final)

if is_valid:
    print("\n✅ All validation checks passed! Dataset is ready for analysis.")
else:
    print("\n⚠️ Some validation issues found. Review before proceeding.")

---

## 📊 Part 8: Exploratory Analysis of Merged Data

Let's explore relationships between bike availability and weather.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10. EXPLORATORY ANALYSIS
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('🔍 Bike Availability vs Weather Conditions', fontsize=16, fontweight='bold')

# 1. Bikes vs Temperature
axes[0, 0].scatter(df_final['temperature_c'], df_final['bikes_available'], 
                   alpha=0.5, color='coral', s=20)
axes[0, 0].set_xlabel('Temperature (°C)')
axes[0, 0].set_ylabel('Bikes Available')
axes[0, 0].set_title('Bike Availability vs Temperature')
axes[0, 0].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(df_final['temperature_c'], df_final['bikes_available'], 1)
p = np.poly1d(z)
axes[0, 0].plot(df_final['temperature_c'].sort_values(), 
                p(df_final['temperature_c'].sort_values()), 
                "r--", alpha=0.8, linewidth=2)

# 2. Bikes vs Precipitation
axes[0, 1].scatter(df_final['precipitation_mm'], df_final['bikes_available'], 
                   alpha=0.5, color='steelblue', s=20)
axes[0, 1].set_xlabel('Precipitation (mm)')
axes[0, 1].set_ylabel('Bikes Available')
axes[0, 1].set_title('Bike Availability vs Precipitation')
axes[0, 1].grid(True, alpha=0.3)

# 3. Bikes vs Wind Speed
axes[1, 0].scatter(df_final['wind_speed_kmh'], df_final['bikes_available'], 
                   alpha=0.5, color='green', s=20)
axes[1, 0].set_xlabel('Wind Speed (km/h)')
axes[1, 0].set_ylabel('Bikes Available')
axes[1, 0].set_title('Bike Availability vs Wind Speed')
axes[1, 0].grid(True, alpha=0.3)

# 4. Time series for one station
station = df_final['station_id'].iloc[0]
station_data = df_final[df_final['station_id'] == station].sort_values('timestamp')

ax4_2 = axes[1, 1].twinx()
axes[1, 1].plot(station_data['timestamp'], station_data['bikes_available'], 
                color='steelblue', linewidth=2, label='Bikes Available')
ax4_2.plot(station_data['timestamp'], station_data['temperature_c'], 
           color='coral', linewidth=2, label='Temperature', alpha=0.7)

axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('Bikes Available', color='steelblue')
ax4_2.set_ylabel('Temperature (°C)', color='coral')
axes[1, 1].set_title(f'Time Series - {station}')
axes[1, 1].grid(True, alpha=0.3)

# Combine legends
lines1, labels1 = axes[1, 1].get_legend_handles_labels()
lines2, labels2 = ax4_2.get_legend_handles_labels()
axes[1, 1].legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

# Calculate correlations
print("\n📊 Correlation Analysis:")
print("=" * 60)
corr_cols = ['bikes_available', 'temperature_c', 'precipitation_mm', 
             'wind_speed_kmh', 'humidity_pct']
correlation_matrix = df_final[corr_cols].corr()
print("\nCorrelation with bikes_available:")
print(correlation_matrix['bikes_available'].sort_values(ascending=False))

### 🧠 Learner Task 2: Create Feature Engineering Ideas

**Your Task**: Based on the merged data, suggest 5 new features that could be useful for predicting bike availability.

Write your ideas below:

### My Feature Ideas:

1. **[Your feature idea 1]**
   - Description: [Why this would be useful]
   - Calculation: [How to compute it]

2. **[Your feature idea 2]**
   - Description: 
   - Calculation:

3. **[Your feature idea 3]**
   - Description:
   - Calculation:

4. **[Your feature idea 4]**
   - Description:
   - Calculation:

5. **[Your feature idea 5]**
   - Description:
   - Calculation:

---

## 💾 Part 9: Save Final Merged Dataset

Save the integrated dataset with proper documentation.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 11. SAVE MERGED DATASET
# ═══════════════════════════════════════════════════════════

# Setup output directory
OUTPUT_DIR = Path('../../data/processed') if 'notebooks' in os.getcwd() else Path('data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Generate filename with timestamp
timestamp = datetime.now().strftime('%Y-%m-%d_%H%M%S')
output_file = OUTPUT_DIR / f'bike_weather_merged_{timestamp}.parquet'

# Save as Parquet (efficient for analysis)
print("💾 Saving merged dataset...")
df_final.to_parquet(output_file, index=False)
print(f"✅ Saved: {output_file.name}")
print(f"   Size: {output_file.stat().st_size / 1024:.1f} KB")

# Also save as CSV for easy inspection
csv_file = output_file.with_suffix('.csv')
df_final.to_csv(csv_file, index=False)
print(f"✅ Saved CSV: {csv_file.name}")

# Create comprehensive metadata
metadata = {
    'dataset_name': 'bike_weather_merged',
    'description': 'Merged bike availability and weather data for Amsterdam',
    'created_date': datetime.now().isoformat(),
    'merge_strategy': 'Aggregated bike data to hourly, exact timestamp match with weather',
    'data_sources': {
        'bike': 'CityBikes API (simulated)',
        'weather': 'Open-Meteo API (simulated)'
    },
    'time_period': {
        'start': df_final['timestamp'].min().isoformat(),
        'end': df_final['timestamp'].max().isoformat(),
        'duration_days': (df_final['timestamp'].max() - df_final['timestamp'].min()).days,
        'frequency': 'hourly'
    },
    'shape': {
        'rows': len(df_final),
        'columns': len(df_final.columns),
        'stations': df_final['station_id'].nunique()
    },
    'columns': {col: str(dtype) for col, dtype in df_final.dtypes.items()},
    'data_quality': {
        'missing_values': int(df_final.isna().sum().sum()),
        'completeness_pct': float((1 - df_final.isna().sum().sum() / df_final.size) * 100)
    },
    'statistics': {
        'avg_bikes_available': float(df_final['bikes_available'].mean()),
        'avg_temperature_c': float(df_final['temperature_c'].mean()),
        'total_precipitation_mm': float(df_final['precipitation_mm'].sum()),
        'avg_wind_speed_kmh': float(df_final['wind_speed_kmh'].mean())
    },
    'correlations': {
        'bikes_temperature': float(df_final['bikes_available'].corr(df_final['temperature_c'])),
        'bikes_precipitation': float(df_final['bikes_available'].corr(df_final['precipitation_mm'])),
        'bikes_wind': float(df_final['bikes_available'].corr(df_final['wind_speed_kmh']))
    }
}

# Save metadata
metadata_file = output_file.with_suffix('.metadata.json')
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Saved metadata: {metadata_file.name}")

print("\n" + "=" * 60)
print("📊 Final Dataset Summary:")
print("=" * 60)
print(f"Filename: {output_file.name}")
print(f"Rows: {metadata['shape']['rows']:,}")
print(f"Columns: {metadata['shape']['columns']}")
print(f"Stations: {metadata['shape']['stations']}")
print(f"Time Period: {metadata['time_period']['duration_days']} days")
print(f"Completeness: {metadata['data_quality']['completeness_pct']:.1f}%")
print(f"\nKey Statistics:")
print(f"  Avg bikes available: {metadata['statistics']['avg_bikes_available']:.2f}")
print(f"  Avg temperature: {metadata['statistics']['avg_temperature_c']:.2f}°C")
print(f"  Total precipitation: {metadata['statistics']['total_precipitation_mm']:.2f}mm")

---

## 📝 Part 10: Summary

### What You've Learned ✅

In this notebook, you:
1. ✅ Understood time alignment challenges in data merging
2. ✅ Implemented two merge strategies (nearest time vs aggregation)
3. ✅ Handled missing data with forward fill and interpolation
4. ✅ Validated merged datasets comprehensively
5. ✅ Explored relationships between bike availability and weather
6. ✅ Saved integrated dataset with documentation
7. ✅ Compared trade-offs between merge approaches

### Key Takeaways 💡

1. **Time alignment is crucial** - Different data sources rarely have perfect timestamps
2. **Choose merge strategy based on use case** - Aggregation vs nearest time matching
3. **Always validate after merging** - Check for missing data, gaps, outliers
4. **Document merge decisions** - Explain what you did and why
5. **Explore relationships early** - Understand correlations before modeling

### Merge Strategy Decision Guide

**Use Nearest Time when:**
- Need to preserve original time resolution
- Working with irregular time intervals
- Want maximum data points for analysis

**Use Aggregation when:**
- Want clean 1:1 alignment
- Different data sources have very different frequencies
- Need to reduce dataset size
- Prefer statistical summaries over raw values

### Next Steps 🚀

You've completed Module 02! Your integrated dataset is ready for:
- **Module 03: Exploration & Profiling** - Deep dive into patterns
- **Module 04: Feature Engineering** - Create predictive features
- **Module 05: Modeling** - Build prediction models

### 🧠 Reflection Questions

1. **Which merge strategy** would you use for real-time prediction vs historical analysis?
2. **How would you handle** data from 100 bike stations instead of 3?
3. **What additional data sources** could improve bike availability predictions?
4. **How would you automate** this merge process to run daily?

**Write your reflections below** ⬇️

### My Reflections

[Your thoughts here]

---

## 📚 References

- [Pandas merge_asof Documentation](https://pandas.pydata.org/docs/reference/api/pandas.merge_asof.html)
- [Time Series Analysis with Pandas](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [Handling Missing Data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Data Integration Best Practices](https://www.analytics-link.com/post/data-integration-best-practices)

---

**🎉 Congratulations!** You've successfully completed Module 02 - Data Acquisition!

**Module 02 Complete** ✅
- M2_01: Amsterdam Bike API ✅
- M2_02: Weather Data API ✅
- M2_03: Data Storage Patterns ✅
- M2_04: Merge Datasets ✅

**Ready for Module 03: Exploration & Profiling!** 🚀